# 05 — Validación de uplift (out-of-sample) e incertidumbre del CATE

Los notebooks anteriores estiman *cuánto* efecto hay. Aquí respondemos dos preguntas de rigor:

1. **¿El modelo ordena bien a los clientes por respuesta?** Lo medimos **fuera de muestra** (train/test) con curvas **Qini** y **AUUC**.
2. **¿Cuánta incertidumbre hay?** Reportamos **intervalos de confianza** del `CausalForestDML` a nivel individuo y por segmento.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from causalml.metrics import plot_gain, plot_qini

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data import load_data
from src.uplift import cate_with_confidence, evaluate_uplift, summarize_cate_ci

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
df = load_data(ROOT / 'data' / 'datos_prueba_tecnica.csv')

## 1. Validación out-of-sample: ¿el modelo rankea bien?

Dividimos control + tratamiento en **train/test** (estratificado por tratamiento), ajustamos T-/X-Learner y Causal Forest en train y evaluamos en el **test** que el modelo no vio. Un Qini/AUUC normalizado alto = el modelo prioriza correctamente a los clientes que más responden; un ranking aleatorio da ~0.

In [ ]:
scores2, scored2 = evaluate_uplift(df, 'trat2', 'ctor', learners=('t', 'x', 'cf'),
                                   n_estimators=200, dml_cv=3, test_size=0.3, random_state=42)
scores1, scored1 = evaluate_uplift(df, 'trat1', 'ctor', learners=('t', 'x', 'cf'),
                                   n_estimators=200, dml_cv=3, test_size=0.3, random_state=42)
print('Trat2 vs ctrl (ctor) — uplift fuera de muestra:')
display(scores2.round(4))
print('Trat1 vs ctrl (ctor) — uplift fuera de muestra:')
display(scores1.round(4))

### Curvas Qini y de ganancia acumulada (mejor modelo para trat2)

La curva del modelo por encima de la diagonal *Random* indica que targetear por uplift captura más conversión que targetear al azar.

In [ ]:
best = scores2.iloc[0]['model']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_qini(scored2[['y', 'w', best]], outcome_col='y', treatment_col='w', ax=axes[0])
axes[0].set_title(f'Qini — {best} (trat2 vs ctrl, ctor)')
plot_gain(scored2[['y', 'w', best]], outcome_col='y', treatment_col='w', ax=axes[1])
axes[1].set_title(f'Ganancia acumulada — {best}')
plt.tight_layout()
plt.show()

**Lectura:** el Causal Forest suele liderar el Qini fuera de muestra. Como el efecto de trat2 es grande, incluso targetear la mitad superior por uplift concentra la mayor parte de los clics incrementales — evidencia de que el *targeting* propuesto en el notebook 03 realmente funciona, no solo en la muestra de entrenamiento.

## 2. Incertidumbre del CATE (IC 95% del Causal Forest)

El punto estimado por sí solo puede sobre-interpretarse. El `CausalForestDML` entrega un intervalo de confianza por cliente; marcamos como **significativo** a quien tiene un IC que excluye el 0.

In [ ]:
ci = cate_with_confidence(df, 'trat2', 'ctor', n_estimators=200, dml_cv=3,
                          alpha=0.05, random_state=42)
display(summarize_cate_ci(ci).to_frame('valor').round(4))

### CATE por edad con intervalo de confianza

Barras de error = media de los límites individuales del IC por segmento (aproximación visual); `frac_sig` = proporción de clientes del segmento con efecto individualmente significativo.

In [ ]:
ci = ci.assign(edad_bin=pd.cut(ci['edad'], [18, 35, 50, 100], labels=['18-35', '36-50', '51+']))
seg = (ci.groupby('edad_bin', observed=True)
         .agg(mean_cate=('cate', 'mean'), ci_lower=('ci_lower', 'mean'),
              ci_upper=('ci_upper', 'mean'), frac_sig=('significant', 'mean'),
              n=('cate', 'size')))
display(seg.round(4))

fig, ax = plt.subplots(figsize=(8, 5))
x = seg.index.astype(str)
ax.errorbar(x, seg['mean_cate'],
            yerr=[seg['mean_cate'] - seg['ci_lower'], seg['ci_upper'] - seg['mean_cate']],
            fmt='o', capsize=6, markersize=8)
ax.axhline(0, color='gray', ls='--')
ax.set_ylabel('CATE (ctor)')
ax.set_title('CATE de trat2 por edad con IC 95%')
plt.tight_layout()
plt.show()

## Conclusión

- **El targeting está validado fuera de muestra:** Qini/AUUC muy por encima del azar, con el Causal Forest a la cabeza.
- **La heterogeneidad es real, pero con matices:** una fracción relevante de clientes tiene un efecto individualmente significativo (IC excluye 0); en otros el efecto no se distingue del ruido, así que conviene priorizar los segmentos de mayor CATE **y** mayor significancia.
- Combinado con el impacto a escala del notebook 04, esto respalda desplegar **trat2** priorizando los segmentos jóvenes / usuarios de app.